In [28]:
import pandas as pd # type: ignore
import os
from google.colab import drive #type: ignore
from google.colab import files #type: ignore
import re

In [14]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
FOLDER_ID = "1gp8vniUcUGgtBPru_Z25qXyJgdrIghis"

# Walk the mounted Drive and find files inside that folder by matching ID is hard,
# so instead: list MyDrive and look for likely folder names, then recurse.
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if "R1_MIS" in f:
            print("FOUND:", os.path.join(root, f))

FOUND: /content/drive/MyDrive/colab_data/R1_MIS_GGN_SEC-48_AUG_2026.xls


In [30]:
path = "/content/drive/MyDrive/colab_data/R1_MIS_GGN_SEC-48_AUG_2026.xls"
df = pd.read_excel(path)

In [49]:
def standardise_columns(df):
    df = df.copy()
    cols = []
    for c in df.columns:
        c = str(c).strip()                                  # trim
        c = c.replace("\n", " ").replace("\t", " ")         # newlines/tabs -> space
        c = re.sub(r"\s+", " ", c)                          # collapse spaces
        c = re.sub(r"[^\w\s]", "", c)                       # drop punctuation
        c = c.strip().lower().replace(" ", "_")             # snake_case
        cols.append(c)
    df.columns = cols
    return df

df = standardise_columns(df)
print(df.columns.tolist())

['s_no', 'date', 'route_no', 'trip_id', 'shift', 'trip_direction', 'imp_id', 'emp_name', 'location', 'zone', 'cab_type', 'rate', 'trip_amount', 'drop', 'remark', 'remark2']


In [60]:
df = df.rename(columns={"trip": "trip_amount"})
df = df.rename(columns={"loginlogout": "trip_direction"})
df = df.rename(columns={"drop": "vehicle_number"})
df["trip_amount"] = (df["rate"] / 44).round(2)
df["route_no"] = df["route_no"].astype("Int64")

In [43]:
def humanise_columns(df):
    """UPPERCASE column names and replace underscores with spaces."""
    df = df.copy()
    df.columns = [
        str(c).strip().upper().replace("_", " ")
        for c in df.columns
    ]
    return df

In [61]:
# df = humanise_columns(df)
df.head()

,s_no,date,route_no,trip_id,shift,trip_direction,imp_id,emp_name,location,zone,cab_type,rate,trip_amount,vehicle_number,remark,remark2
0,1,2026-08-03,8012,1957685,17:30:00,LOGOUT,206241,MANISH,MADIPUR,WEST,SMALL,29000,659.09,1335,--,NaN
1,2,2026-08-03,8013,1957686,17:30:00,LOGOUT,10041060,BRAJESH,MOHAN GARGAN,WEST,BIG,34000,772.73,4001,--,NaN
2,3,2026-08-03,8014,1957687,17:30:00,LOGOUT,10051905,PRATEEK,ROHINI,WEST,SMALL,29000,659.09,401,--,NaN
3,4,2026-08-03,8015,1957688,17:30:00,LOGOUT,10014813,RAJINDRA,NANGLI METRO,WEST,SMALL,25000,568.18,4538,--,NaN
4,5,2026-08-03,8016,1957689,17:30:00,LOGOUT,202018,PANKAJ,UTTAM NAGAR,WEST,SMALL,25000,568.18,5511,--,NaN


In [62]:
print(df.to_string())

      s_no       date  route_no  trip_id     shift trip_direction    imp_id                  emp_name                         location     zone cab_type    rate  trip_amount vehicle_number    remark  remark2
0        1 2026-08-03      8012  1957685  17:30:00         LOGOUT    206241                    MANISH                          MADIPUR     WEST    SMALL   29000       659.09           1335        --      NaN
1        2 2026-08-03      8013  1957686  17:30:00         LOGOUT  10041060                   BRAJESH                     MOHAN GARGAN     WEST      BIG   34000       772.73           4001        --      NaN
2        3 2026-08-03      8014  1957687  17:30:00         LOGOUT  10051905                   PRATEEK                           ROHINI     WEST    SMALL   29000       659.09            401        --      NaN
3        4 2026-08-03      8015  1957688  17:30:00         LOGOUT  10014813                  RAJINDRA                     NANGLI METRO     WEST    SMALL   25000       5